# CaPy v2 — Training & Evaluation

Tri-modal contrastive learning for drug discovery.

## How to use this notebook

- **To pull new code:** Re-run **Cell 1** (Setup). It does `git pull` automatically — no page refresh needed.
- **First time?** Run cells 1–4 in order (Setup → Download → Preprocess → Verify).
- **New runtime?** Run Cells 1–4 in order (Setup → Download → Preprocess → Verify), then jump to your phase.
- **Same runtime, pulling new code?** Re-run Cell 1 only, then jump to your phase.
- **`--resume` flag:** Ablation cells use `--resume` so they skip already-completed runs (checks `results/ablation_runs.jsonl`, tracked in git).

## Notebook map

| Section | Cells | Purpose |
|---------|-------|---------|
| **Setup** | 1–5 | Clone/pull repo, install deps, download data, preprocess, W&B login |
| **Phase 1** | 6–10 | Bi-modal mol↔morph baseline (single + multi-seed) |
| **Phase 2** | 11–15 | Additional bi-modal baselines + tri-modal (single seed) |
| **Phase 2 — Sweeps** | 16–27 | Mol-direction remediation sweeps (already complete) |
| **Phase 3 — Ablations** | 28–31 | Full 8-config × 5-seed matrix + summary statistics |
| **Phase 3 — Report** | 32–33 | Full evaluation report (UMAP, retrieval tables, MOA clustering) |

## Phase status
- **Phase 1:** PASSED — compound R@10 > 10% across 4 seeds
- **Phase 2:** PASSED — T1 morph↔expr +10pp over B6; S2b locked as default
- **Phase 3:** COMPLETE — 24/24 runs, Partial Success (PRD §4.1)

In [ ]:
# ── Cell 1: Setup (clone/pull + install) ──────────────────────────────
# Re-run this cell to pull the latest code from GitHub.
import os

if not os.path.exists("/content/CaPy-v2"):
    !git clone https://github.com/ogngnaoh/CaPy-v2.git
else:
    %cd /content/CaPy-v2
    !git pull
    %cd /content

%cd /content/CaPy-v2
!pip install -e ".[dev]" -q

# Verify install
import torch
print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Cell 2: Download data ─────────────────────────────────────────────
# Downloads morphology (CellProfiler), expression (L1000), and metadata.
# Skips files that already exist. ~150 MB total.
!python3 scripts/download.py

In [ ]:
# ── Cell 3: Preprocess ────────────────────────────────────────────────
# QC, normalize, scaffold split → data/processed/{train,val,test}.parquet
# Only needs to run once (unless raw data changes).
!python3 scripts/preprocess.py

In [ ]:
# ── Cell 4: Verify processed data ─────────────────────────────────────
# Quick sanity check — shows split sizes and feature counts.
import json
from pathlib import Path

import pandas as pd

processed = Path("data/processed")
for split in ["train", "val", "test"]:
    df = pd.read_parquet(processed / f"{split}.parquet")
    print(f"{split}: {len(df)} treatments, {df.shape[1]} columns")

with open(processed / "feature_columns.json") as f:
    cols = json.load(f)
print(f"\nMorph features: {len(cols['morph_features'])}")
print(f"Expr features: {len(cols['expr_features'])}")

In [ ]:
# ── Cell 5: W&B login (optional) ──────────────────────────────────────
# Skip this cell if you don't want W&B logging.
# Training works fine without it (wandb=false by default).
import wandb
wandb.login()

---
## Phase 1: Bi-Modal Mol↔Morph Baseline ✅

**Gate criteria:** compound-level R@10 > 10% AND alignment < 1.5

Run single seed first to verify, then multi-seed for robustness.

> **Status: PASSED** — mean 13.4% ± 0.9% across 4 seeds.

In [ ]:
# ── Cell 7: Phase 1 gate — bi-modal mol↔morph (single seed) ──────────
!python3 scripts/train.py model=bi_mol_morph seed=42

### Multi-Seed Validation

Run 3 additional seeds to confirm Phase 1 gate result is robust.
All seeds should pass: compound R@10 > 10% AND alignment < 1.5.

In [ ]:
# ── Cell 9: Multi-seed validation (seeds 123, 456, 789) ──────────────
for seed in [123, 456, 789]:
    print(f"\n{'='*60}")
    print(f"Training seed={seed}")
    print(f"{'='*60}")
    !python3 scripts/train.py model=bi_mol_morph seed={seed}

In [ ]:
# ── Cell 10: Compare multi-seed Phase 1 results ──────────────────────
import torch

print(f"{'Seed':<8} {'Epoch':<8} {'compound R@10':<16} {'Alignment':<12} {'Uniform mol':<14} {'Uniform morph':<14} {'Gate'}")
print("-" * 90)

for seed in [42, 123, 456, 789]:
    path = f"checkpoints/bi_mol_morph_seed{seed}.pt"
    try:
        ckpt = torch.load(path, weights_only=False, map_location="cpu")
        metrics = ckpt.get("metrics", {})
        r10 = ckpt.get("best_metric", metrics.get("compound/mean_R@10", -1))
        align = metrics.get("align_mol_morph", -1)
        u_mol = metrics.get("uniform_mol", -1)
        u_morph = metrics.get("uniform_morph", -1)
        epoch = ckpt.get("epoch", -1)
        gate = "PASS" if (r10 > 0.10 and align < 1.5) else "FAIL"
        print(f"{seed:<8} {epoch:<8} {r10:<16.4f} {align:<12.4f} {u_mol:<14.4f} {u_morph:<14.4f} {gate}")
    except FileNotFoundError:
        print(f"{seed:<8} not found (not trained yet)")

print("\nGate: compound R@10 > 10% AND alignment < 1.5")

---
## Phase 2: Bi-Modal Baselines + Tri-Modal ✅

Train remaining bi-modal configs (B5, B6) and tri-modal (T1) with single seed.

> **Status: PASSED** — T1 morph↔expr R@10 = 84.8% (B6 = 75.1%, +10pp).

In [ ]:
# ── Cell 12: B5 — mol↔expr bi-modal ──────────────────────────────────
!python3 scripts/train.py model=bi_mol_expr seed=42

In [ ]:
# ── Cell 13: B6 — morph↔expr bi-modal ────────────────────────────────
!python3 scripts/train.py model=bi_morph_expr seed=42

In [ ]:
# ── Cell 14: T1 — tri-modal (all 3 modalities) ───────────────────────
!python3 scripts/train.py model=tri_modal seed=42

In [ ]:
# ── Cell 15: Compare all configs (single seed) ───────────────────────
import torch

configs = [
    ("bi_mol_morph", 42),
    ("bi_mol_expr", 42),
    ("bi_morph_expr", 42),
    ("tri_modal", 42),
]

print(f"{'Config':<20} {'Epoch':<8} {'Best R@10':<12}")
print("-" * 42)

for config, seed in configs:
    path = f"checkpoints/{config}_seed{seed}.pt"
    try:
        ckpt = torch.load(path, weights_only=False, map_location="cpu")
        r10 = ckpt.get("best_metric", -1)
        epoch = ckpt.get("epoch", -1)
        print(f"{config:<20} {epoch:<8} {r10:<12.4f}")
    except FileNotFoundError:
        print(f"{config:<20} not found")

### Phase 2 — Quick multi-seed (B4-T1 × 5 seeds)

Runs the 4 trained configs × 5 seeds = 20 runs. `--resume` skips seeds with existing checkpoints.

In [ ]:
# ── Cell 17: Multi-seed ablation (B4-T1 only, skips B0-B3) ───────────
# Use this if you want to run just the trained configs without baselines.
!python3 scripts/run_ablations.py --matrix core --resume --configs B4 B5 B6 T1

---
## Phase 2 Remediation: Mol-Direction Sweeps ✅

T1 morph↔expr R@10 = 84.8% (beats B6 by +10pp), but mol-containing directions stuck at ~12%.
These sweeps tested strategies to improve mol directions.

> **Status: COMPLETE** — S2b (per-pair SigLIP + 2x mol weights) locked as default T1 config.
> Cells 19–27 are kept for reference but don't need to be re-run.

In [ ]:
# ── Cell 19: S1 — per-pair SigLIP baseline (equal weights) ───────────
# Isolates effect of separate temp/bias per modality pair.
!python3 scripts/train.py model=tri_modal seed=42

In [ ]:
# ── Cell 20: S2a — downweight morph↔expr ─────────────────────────────
!python3 scripts/train.py model=tri_modal seed=42 \
    training.loss.pair_weights.mol_morph=1 \
    training.loss.pair_weights.mol_expr=1 \
    training.loss.pair_weights.morph_expr=0.3

In [ ]:
# ── Cell 21: S2b — upweight mol pairs 2x (WINNER → locked as default) ─
!python3 scripts/train.py model=tri_modal seed=42 \
    training.loss.pair_weights.mol_morph=2 \
    training.loss.pair_weights.mol_expr=2 \
    training.loss.pair_weights.morph_expr=1

In [ ]:
# ── Cell 22: S2c — upweight mol pairs 3x, downweight morph↔expr ──────
!python3 scripts/train.py model=tri_modal seed=42 \
    training.loss.pair_weights.mol_morph=3 \
    training.loss.pair_weights.mol_expr=3 \
    training.loss.pair_weights.morph_expr=0.5

In [ ]:
# ── Cell 23: S3a — discriminative LR (mol 2x, morph/expr 0.5x) ───────
!python3 scripts/train.py model=tri_modal seed=42 \
    training.optimizer.modality_lr_mult.mol=2.0 \
    training.optimizer.modality_lr_mult.morph=0.5 \
    training.optimizer.modality_lr_mult.expr=0.5

In [ ]:
# ── Cell 24: S3b — discriminative LR (mol 3x, morph/expr 0.3x) ───────
!python3 scripts/train.py model=tri_modal seed=42 \
    training.optimizer.modality_lr_mult.mol=3.0 \
    training.optimizer.modality_lr_mult.morph=0.3 \
    training.optimizer.modality_lr_mult.expr=0.3

In [ ]:
# ── Cell 25: S4 — curriculum (ramp mol weights over 50 epochs) ────────
!python3 scripts/train.py model=tri_modal seed=42 \
    training.loss.curriculum.enabled=true \
    training.loss.curriculum.warmup_epochs=50

In [ ]:
# ── Cell 26: S5 — staged (morph↔expr 100ep, then freeze + add mol) ───
!python3 scripts/train.py model=tri_modal training=staged seed=42

In [ ]:
# ── Cell 27: Compare sweep results — per-direction R@10 ──────────────
!python3 scripts/analyze_checkpoints.py

---
## Phase 3: Ablation Matrix ✅

**8 configs, 24 total runs** (4 baselines × 1 seed + 4 trained × 5 seeds, written to `results/ablation_runs.jsonl`):

| Config | Type | Seeds | What |
|--------|------|-------|------|
| B0 | Baseline | 1 | Random 256-dim embeddings (absolute floor) |
| B1 | Baseline | 1 | Raw ECFP features, no training |
| B2 | Baseline | 1 | Raw CellProfiler features, no training |
| B3 | Baseline | 1 | Raw L1000 features, no training |
| B4 | Trained | 5 | Bi-modal mol↔morph |
| B5 | Trained | 5 | Bi-modal mol↔expr |
| B6 | Trained | 5 | Bi-modal morph↔expr |
| T1 | Trained | 5 | Tri-modal (CaPy core claim) |

**`--resume` is safe to re-run.** It checks `ablation_runs.jsonl` — any (config, seed) already there gets skipped. No duplicate entries.

**Estimated runtime:** B0-B3 evaluate in ~1 min total. B4-T1 training: ~15 min/run × 20 runs ≈ 5 hours on H100. With `--resume`, only missing runs execute.

In [ ]:
# ── Cell 29: Run full ablation matrix (B0-B3 baselines + B4-T1 training)
# --resume skips any (config, seed) already in ablation_runs.jsonl.
# B0-B3: evaluates raw features / random embeddings (~1 min total).
# B4-T1: trains 20 models (~5 hours on H100, skips existing).
!python3 scripts/run_ablations.py --matrix core --resume

In [ ]:
# ── Cell 30: Generate ablation summary + statistical tests ───────────
# Reads results/ablation_runs.jsonl → produces:
#   - results/ablation_summary.csv    (8-row config × metrics table)
#   - results/ablation_comparison.tex (LaTeX table with p-values)
#   - results/ablation_barplot.png    (bar chart: 4 grey + 3 blue + 1 green)
#   - stdout: formatted comparison table with Welch's t-test results
!python3 -m scripts.summarize_ablations

In [ ]:
# ── Cell 31: Detailed per-direction results table ────────────────────
# Shows all 6 retrieval directions per config × seed from checkpoints.
import torch
from pathlib import Path

configs = {
    "B4": "bi_mol_morph",
    "B5": "bi_mol_expr",
    "B6": "bi_morph_expr",
    "T1": "tri_modal",
}
seeds = [42, 123, 456, 789, 1024]

directions = ["mol→morph", "morph→mol", "mol→expr", "expr→mol", "morph→expr", "expr→morph"]
print(f"{'Config':<6} {'Seed':<6} {'Epoch':<6} {'mean R@10':<10} {'cmpd R@10':<10} " + " ".join(f"{d:<12}" for d in directions))
print("-" * 120)

for config_id, name in configs.items():
    for seed in seeds:
        path = Path(f"checkpoints/{name}_seed{seed}.pt")
        if not path.exists():
            print(f"{config_id:<6} {seed:<6} {'—'}")
            continue
        ckpt = torch.load(path, weights_only=False, map_location="cpu")
        m = ckpt.get("metrics", {})
        epoch = ckpt.get("epoch", -1)
        mean_r10 = m.get("mean_R@10", -1)
        cmpd_r10 = ckpt.get("best_metric", m.get("compound/mean_R@10", -1))

        dir_vals = []
        for pair in [("mol", "morph"), ("morph", "mol"), ("mol", "expr"), ("expr", "mol"), ("morph", "expr"), ("expr", "morph")]:
            key = f"compound/{pair[0]}->{pair[1]}/R@10"
            val = m.get(key, m.get(f"{pair[0]}->{pair[1]}/R@10", -1))
            dir_vals.append(f"{val:<12.4f}" if val >= 0 else f"{'N/A':<12}")

        print(f"{config_id:<6} {seed:<6} {epoch:<6} {mean_r10:<10.4f} {cmpd_r10:<10.4f} " + " ".join(dir_vals))
    print()

# Show JSONL line count
jsonl = Path("results/ablation_runs.jsonl")
if jsonl.exists():
    print(f"results/ablation_runs.jsonl: {sum(1 for _ in open(jsonl))} lines")

---
## Phase 3: Full Evaluation Report

Run full evaluation on best T1 checkpoint (FR-8.4).
Generates: retrieval table (CSV + LaTeX), UMAP plots, similarity heatmap,
training curves, MOA clustering metrics, and formatted summary table.

In [ ]:
# ── Cell 33: Full evaluation report on best T1 checkpoint ────────────
# Outputs to results/:
#   - retrieval_table.csv, retrieval_table.tex
#   - umap_*.png (per modality)
#   - similarity_heatmap.png
#   - training_curves.png
#   - MOA clustering metrics + summary table (stdout)
!python3 scripts/evaluate.py --checkpoint checkpoints/tri_modal_seed42.pt --full \
    --data-dir data/processed --output-dir results